# Week 1 — Random Forest Baseline (canonical user split)

This notebook trains the Week 1 Random Forest baseline for binary insider-threat classification, **reusing the canonical 60/20/20 user-level split** that already exists in the project (`data/user_splits/`)

**Protocol (strict):**
1. Load the saved split (no re-splitting, no re-seeding of a new split).
2. Train the Random Forest **only on train users**.
3. Perform threshold calibration **only on validation users**.
4. Evaluate **once** on the untouched held-out test users.


## 1. Setup

Resolve project root; load the processed features produced by `week1_eda.ipynb` and the saved user split.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
import warnings
warnings.filterwarnings('ignore')

# Resolve project root (portability convention)
_curr = Path.cwd()
while not (_curr / 'pyproject.toml').exists() and _curr != _curr.parent:
    _curr = _curr.parent
PROJECT_ROOT = _curr
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
SPLIT_DIR = DATA_DIR / 'user_splits'

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\insider_threat_detection_project2


## 2. Load Processed Features & Canonical Split

### 2.1 Features

Load the clean feature matrix saved by `week1_eda.ipynb` (`data/processed/features_clean.parquet`), which retains `user` and the binary `insider` target.

In [2]:
X = pd.read_parquet(PROCESSED_DIR / 'features_clean.parquet')
print(f"Processed features: {X.shape}")
print(f"Columns: {X.columns.tolist()[:8]} ...")

Processed features: (330452, 504)
Columns: ['user', 'isweekday', 'isweekend', 'role', 'b_unit', 'f_unit', 'dept', 'team'] ...


### 2.2 Canonical user split

Load the **exact** train/validation/test users from the saved split files. 

In [3]:
train_users = np.load(SPLIT_DIR / 'train_users.npy')
val_users   = np.load(SPLIT_DIR / 'val_users.npy')
test_users  = np.load(SPLIT_DIR / 'test_users.npy')

print(f"Train users: {len(train_users)} | Val users: {len(val_users)} | Test users: {len(test_users)}")
print(f"Overlap check (must all be 0): "
      f"train/val={len(set(train_users)&set(val_users))}, "
      f"train/test={len(set(train_users)&set(test_users))}, "
      f"val/test={len(set(val_users)&set(test_users))}")

# --- Partition the feature matrix by the canonical split ---
train_df = X[X['user'].isin(train_users)]
val_df   = X[X['user'].isin(val_users)]
test_df  = X[X['user'].isin(test_users)]

def drop_meta(df):
    return df.drop(columns=['user'])

# Features are all numeric; no missing values (verified in EDA)
feature_cols = [c for c in X.columns if c not in ('user', 'insider')]

X_train = drop_meta(train_df)[feature_cols]
y_train = train_df['insider'].astype(int).values
X_val   = drop_meta(val_df)[feature_cols]
y_val   = val_df['insider'].astype(int).values
X_test  = drop_meta(test_df)[feature_cols]
y_test  = test_df['insider'].astype(int).values

print(f"\nTrain: {X_train.shape[0]} rows ({len(train_users)} users) | pos={y_train.sum()}")
print(f"Val  : {X_val.shape[0]} rows ({len(val_users)} users) | pos={y_val.sum()}")
print(f"Test : {X_test.shape[0]} rows ({len(test_users)} users) | pos={y_test.sum()}")

Train users: 600 | Val users: 200 | Test users: 200
Overlap check (must all be 0): train/val=0, train/test=0, val/test=0

Train: 199205 rows (600 users) | pos=574
Val  : 65567 rows (200 users) | pos=191
Test : 65680 rows (200 users) | pos=201


## 3. Train Random Forest (train users only)

Hyperparameters are **unchanged** from the existing Week-1 baseline: `n_estimators=200`, `class_weight='balanced'`, `random_state=42`. Only the split source has changed (canonical saved split instead of a locally-generated one).

In [4]:
model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

print("Fitting Random Forest on TRAIN users only...")
model.fit(X_train, y_train)
print("[OK] Model training complete.")

Fitting Random Forest on TRAIN users only...
[OK] Model training complete.


## 4. Threshold Calibration (validation users only)

The Random Forest outputs probabilities. The project's protocol uses a decision threshold (the existing baseline used 0.5 implicitly). We calibrate a **probability threshold on the validation set** (maximizing Youden's J = Recall − FPR, the criterion used across the project), then freeze it before any test evaluation.

In [5]:
from sklearn.metrics import recall_score

# Probability scores on validation only
y_prob_val = model.predict_proba(X_val)[:, 1]

# Sweep a threshold on VALIDATION only to maximize Youden's J
best_j, best_t = -1.0, 0.5
for t in np.arange(0.05, 0.96, 0.05):
    preds = (y_prob_val > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    rec = recall_score(y_val, preds, zero_division=0)
    j = rec - fpr
    if j > best_j:
        best_j, best_t = j, t

THRESHOLD = float(best_t)
print(f"[val] Youden's-J optimal threshold: {THRESHOLD:.2f} (J={best_j:.4f})")
print("Threshold FROZEN — will be applied once to the held-out test set.")

[val] Youden's-J optimal threshold: 0.05 (J=0.8307)
Threshold FROZEN — will be applied once to the held-out test set.


## 5. Evaluate on Held-Out Test (once)

Apply the **frozen** threshold to the untouched test users and report metrics. Test is touched only here.

In [6]:
y_prob_test = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob_test > THRESHOLD).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_prob_test)

print("=== Held-Out Test Metrics (frozen threshold {:.2f}) ===".format(THRESHOLD))
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

print("\nConfusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

=== Held-Out Test Metrics (frozen threshold 0.05) ===
Accuracy : 0.9900
Precision: 0.1812
Recall   : 0.6418
F1-score : 0.2826
ROC-AUC  : 0.9410

Confusion Matrix (rows=true, cols=pred):
[[64896   583]
 [   72   129]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     65479
           1       0.18      0.64      0.28       201

    accuracy                           0.99     65680
   macro avg       0.59      0.82      0.64     65680
weighted avg       1.00      0.99      0.99     65680



## 6. Summary

* Reused the canonical `data/user_splits/` split — no new split generated, seed unchanged, stratification untouched.
* Trained RF on the 60% train users only, with the original hyperparameters.
* Calibrated the decision threshold on validation users only (Youden's J).
* Evaluated **once** on the held-out test users.

The exact users here are identical to those used by Tier-1 / Tier-2 / Tier-3, making every experiment directly comparable.